# Prognoser Runner — Parameterized Survival Model

Edit the `EXPERIMENT` config cell below, then run all cells.

Methods:
- `km` — Kaplan-Meier (population-level, no covariates)
- `cox_clinical` — Cox PH with age/sex/MMSE/CDR/ApoE4
- `cox_embedding` — Cox PH with 64-dim GAAE embedding only
- `cox_combined` — Cox PH with clinical + embedding
- `rsf` — Random Survival Forest (clinical + embedding)
- `deepsurv` — DeepSurv neural Cox (stretch — requires pycox)

Network combos: `dmn`/`hippo`/`limbic`/`dan`/`dmn_hippo`/`dmn_limbic`/`dmn_limbic_hippo`/`all_combined`.

In [ ]:
# ── EXPERIMENT CONFIG ─────────────────────────────────────────────────────────
EXPERIMENT = {
    "network_combo": "dmn_hippo",
    "data_version": "__v8__",
    "file_suffix": "_dmn_hippo_correlation_matrix_z_transformed.npz",
    "method": "cox_clinical",
        # km | cox_clinical | cox_embedding | cox_combined | rsf | deepsurv
    "feature_set": "clinical",       # clinical | embedding | clinical+embedding
    "eval_times": [12, 24, 36, 48, 60, 72],
    "penalizer": 0.05,                # Cox penalizer
    "pca_components": 16,             # for embedding methods
    "rsf_n_estimators": 200,
    "rsf_min_samples_leaf": 5,
    "random_state": 42,
}
# ─────────────────────────────────────────────────────────────────────────────

import sys, os, json
from pathlib import Path
from datetime import datetime

REPO_ROOT = Path('/mnt/e/fyassine/ad-early-detection')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

COHORTS_CSV = REPO_ROOT / 'DATA' / 'DELCODE' / '__v3__' / 'metadata' / 'cohorts.csv'
SPLITS_DIR = REPO_ROOT / 'DATA' / 'DELCODE' / '__v3__' / 'metadata' / 'splits_gec'
EMBEDDINGS_CACHE = REPO_ROOT / 'PROGNOSER' / 'notebooks' / '_embeddings_cache_'
CHECKPOINT_ROOT = REPO_ROOT / 'PROGNOSER' / 'notebooks' / f'checkpoints_prognoser_{EXPERIMENT["network_combo"]}'
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Experiment: {EXPERIMENT["network_combo"]} | Method: {EXPERIMENT["method"]}')
print(f'Checkpoint dir: {CHECKPOINT_ROOT}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PROGNOSER.common.survival_table import build_survival_table, make_xte
from PROGNOSER.common.metrics import evaluate_model
from PROGNOSER.model.kaplan_meier import KMBaseline
from PROGNOSER.model.cox import CoxPHWrapper

sns.set_theme(style='whitegrid')
print('Imports OK')

## Build Survival Tables

In [ ]:
train_table = build_survival_table(str(COHORTS_CSV), splits_dir=str(SPLITS_DIR), split='train')
val_table   = build_survival_table(str(COHORTS_CSV), splits_dir=str(SPLITS_DIR), split='val')
test_table  = build_survival_table(str(COHORTS_CSV), splits_dir=str(SPLITS_DIR), split='test')

for name, t in [('train', train_table), ('val', val_table), ('test', test_table)]:
    print(f'{name}: n={len(t):3d}  events={int(t.event_observed.sum()):3d}  '
          f'rate={t.event_observed.mean():.0%}  median_T={t.duration.median():.0f}m')

## Load Cached GAAE Embeddings (if needed)

In [ ]:
needs_embeddings = EXPERIMENT['method'] in ('cox_embedding', 'cox_combined', 'rsf', 'deepsurv') \
                   and EXPERIMENT['feature_set'] != 'clinical'

embeddings_df = None
if needs_embeddings:
    cache_file = EMBEDDINGS_CACHE / f"{EXPERIMENT['network_combo']}_baseline_embeddings.parquet"
    if not cache_file.exists():
        raise FileNotFoundError(
            f'Embeddings cache not found at {cache_file}. Run:\n'
            f'  python -m PROGNOSER.src.build_baseline_embeddings --combo {EXPERIMENT["network_combo"]}'
        )
    embeddings_df = pd.read_parquet(cache_file)
    print(f'Loaded embeddings: shape={embeddings_df.shape}')
    print(f'Subjects with embeddings: {len(embeddings_df)}')

    # Merge into survival tables
    def _merge(table):
        merged = table.merge(embeddings_df, left_on='subject_id', right_index=True, how='inner')
        return merged.reset_index(drop=True)
    train_table = _merge(train_table)
    val_table   = _merge(val_table)
    test_table  = _merge(test_table)
    print(f'After merge: train={len(train_table)}, val={len(val_table)}, test={len(test_table)}')
else:
    print('Method is clinical-only — skipping embedding load.')

## Build Feature Matrices

In [ ]:
clinical_cols = ['age', 'sex', 'mmstot', 'cdrglobal', 'apoe4']
embedding_cols = [c for c in (embeddings_df.columns if embeddings_df is not None else []) if c.startswith('z_')]

if EXPERIMENT['feature_set'] == 'clinical':
    feature_cols = clinical_cols
elif EXPERIMENT['feature_set'] == 'embedding':
    feature_cols = embedding_cols
elif EXPERIMENT['feature_set'] == 'clinical+embedding':
    feature_cols = clinical_cols + embedding_cols
else:
    raise ValueError(f"Unknown feature_set: {EXPERIMENT['feature_set']}")

X_tr, T_tr, E_tr, used_tr = make_xte(train_table, feature_cols)
X_va, T_va, E_va, used_va = make_xte(val_table, feature_cols)
X_te, T_te, E_te, used_te = make_xte(test_table, feature_cols)

print(f'feature_cols ({len(feature_cols)}): {feature_cols[:8]}{"..." if len(feature_cols) > 8 else ""}')
print(f'X_train shape: {X_tr.shape}  events: {E_tr.sum()}/{len(E_tr)}')
print(f'X_val   shape: {X_va.shape}  events: {E_va.sum()}/{len(E_va)}')
print(f'X_test  shape: {X_te.shape}  events: {E_te.sum()}/{len(E_te)}')

## Instantiate & Fit Model

In [ ]:
method = EXPERIMENT['method']

if method == 'km':
    model = KMBaseline().fit(X_tr, T_tr, E_tr)
elif method == 'cox_clinical':
    model = CoxPHWrapper.with_clinical_features(penalizer=EXPERIMENT['penalizer']).fit(X_tr, T_tr, E_tr)
elif method == 'cox_embedding':
    latent_dim = len(embedding_cols)
    model = CoxPHWrapper.with_embedding_features(
        latent_dim=latent_dim,
        use_pca=True,
        pca_components=min(EXPERIMENT['pca_components'], latent_dim, X_tr.shape[0] - 1),
        penalizer=EXPERIMENT['penalizer'],
    ).fit(X_tr, T_tr, E_tr)
elif method == 'cox_combined':
    latent_dim = len(embedding_cols)
    model = CoxPHWrapper.with_clinical_plus_embedding(
        latent_dim=latent_dim,
        use_pca=True,
        pca_components=min(EXPERIMENT['pca_components'], X_tr.shape[1], X_tr.shape[0] - 1),
        penalizer=EXPERIMENT['penalizer'],
    ).fit(X_tr, T_tr, E_tr)
elif method == 'rsf':
    from PROGNOSER.model.rsf import RSFWrapper
    model = RSFWrapper(
        feature_columns=feature_cols,
        n_estimators=EXPERIMENT['rsf_n_estimators'],
        min_samples_leaf=EXPERIMENT['rsf_min_samples_leaf'],
        random_state=EXPERIMENT['random_state'],
    ).fit(X_tr, T_tr, E_tr)
elif method == 'deepsurv':
    from PROGNOSER.model.deepsurv import DeepSurvWrapper
    model = DeepSurvWrapper(
        feature_columns=feature_cols,
        random_state=EXPERIMENT['random_state'],
    ).fit(X_tr, T_tr, E_tr, X_val=X_va, T_val=T_va, E_val=E_va)
else:
    raise ValueError(f'Unknown method: {method}')

print(f'Fitted: {model.method_name}')

## Evaluate

In [ ]:
eval_times = np.array(EXPERIMENT['eval_times'], dtype=float)

metrics = {
    'train': evaluate_model(model, X_tr, T_tr, E_tr, X_tr, T_tr, E_tr, eval_times=eval_times),
    'val':   evaluate_model(model, X_tr, T_tr, E_tr, X_va, T_va, E_va, eval_times=eval_times),
    'test':  evaluate_model(model, X_tr, T_tr, E_tr, X_te, T_te, E_te, eval_times=eval_times),
}

print('=' * 60)
print(f'{"split":<6} {"C-index":>10} {"IBS":>10} {"AUC@24":>8} {"AUC@36":>8} {"AUC@60":>8}')
print('-' * 60)
for split_name, m in metrics.items():
    auc = m.get('auc', {})
    print(
        f'{split_name:<6} '
        f'{m["c_index"]:>10.4f} '
        f'{m["ibs"]:>10.4f} '
        f'{auc.get(24, float("nan")):>8.3f} '
        f'{auc.get(36, float("nan")):>8.3f} '
        f'{auc.get(60, float("nan")):>8.3f}'
    )
print('=' * 60)

## Plot Predicted Survival Curves

In [ ]:
if method != 'km':
    times_plot = np.linspace(0, max(60, T_tr.max()), 30)
    surv = model.predict_survival(X_te, times_plot)
    risk_te = model.predict_risk(X_te)
    median_risk = np.median(risk_te)
    high_idx = np.where(risk_te >= median_risk)[0]
    low_idx = np.where(risk_te < median_risk)[0]

    fig, ax = plt.subplots(figsize=(9, 5))
    if len(high_idx) > 0:
        ax.plot(times_plot, surv[high_idx].mean(axis=0), color='red', lw=2, label=f'High risk (n={len(high_idx)})')
    if len(low_idx) > 0:
        ax.plot(times_plot, surv[low_idx].mean(axis=0), color='green', lw=2, label=f'Low risk (n={len(low_idx)})')
    ax.set_xlabel('Months from baseline')
    ax.set_ylabel('Predicted P(no AD)')
    ax.set_title(f'{method} | {EXPERIMENT["network_combo"]} | test set predicted survival by risk stratum')
    ax.legend()
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()

## Save Run Artifacts

In [ ]:
run_timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
run_name = f'{method}_{EXPERIMENT["feature_set"].replace("+", "-")}_{run_timestamp}'
run_dir = CHECKPOINT_ROOT / run_name
run_dir.mkdir(parents=True, exist_ok=True)

if method != 'km':
    model.save(str(run_dir / f'model_{run_name}.joblib'))

# Test predictions
risk_te = model.predict_risk(X_te)
preds_df = used_te[['subject_id', 'duration', 'event_observed']].copy()
preds_df['risk'] = risk_te
preds_df.to_csv(run_dir / 'predictions_test.csv', index=False)

run_summary = {
    'run_name': run_name,
    'timestamp': run_timestamp,
    'experiment': EXPERIMENT,
    'method': method,
    'feature_set': EXPERIMENT['feature_set'],
    'n_features': len(feature_cols),
    'feature_columns': feature_cols,
    'n_train': int(len(X_tr)),
    'n_val':   int(len(X_va)),
    'n_test':  int(len(X_te)),
    'n_events_test': int(E_te.sum()),
    'metrics': metrics,
    'eval_times': EXPERIMENT['eval_times'],
}
with open(run_dir / 'run_summary.json', 'w') as f:
    json.dump(run_summary, f, indent=2, default=str)

print(f'Saved: {run_dir}')
print(f'  - run_summary.json')
print(f'  - predictions_test.csv')
if method != 'km':
    print(f'  - model_{run_name}.joblib')
print(f'\nTest C-index: {metrics["test"]["c_index"]:.4f}')